# QA Generator - Response Model Dataset

Generate pasangan (pertanyaan + KG context, jawaban NL ideal) untuk fine-tuning response model.

**Strategi:**
1. Dari Task 2 ground truth: eksekusi Cypher -> format KG results -> LLM generate jawaban
2. Dari document chunks: LLM generate QA -> LLM verifier (Rompis 2025)

**Prerequisites:**
- Neo4j running di `bolt://localhost:7687`
- Task 2 CSV ready (`finetuning/query_model/data/training_data.csv`)
- `GEMINI_API_KEY` di `.env`

## 0. Setup

In [ ]:
import os, sys, json
from dotenv import load_dotenv

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

load_dotenv(os.path.join(ROOT_DIR, '.env'), override=True)

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'passwd123')

QUERY_DATA_PATH = os.path.join(ROOT_DIR, 'finetuning', 'query_model', 'data', 'training_data.csv')
CHUNKS_DIR = os.path.join(ROOT_DIR, 'data', 'processed')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'finetuning', 'response_model', 'data')

print(f"Root: {ROOT_DIR}")
print(f"Gemini API: {'set' if GEMINI_API_KEY else 'NOT SET'}")
print(f"Query data: {'EXISTS' if os.path.exists(QUERY_DATA_PATH) else 'NOT FOUND'}")
print(f"Chunks dir: {'EXISTS' if os.path.isdir(CHUNKS_DIR) else 'NOT FOUND'}")

In [ ]:
# Connect to Neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    cnt = session.run("MATCH (n) RETURN count(n) AS cnt").single()['cnt']
print(f"Neo4j OK! {cnt} nodes")

## 1. Strategy 1: From Task 2 Query Results

In [ ]:
from finetuning.response_model.generate_training_data import generate_from_query_results

query_samples = generate_from_query_results(
    driver=driver,
    query_data_path=QUERY_DATA_PATH,
    api_key=GEMINI_API_KEY,
    max_samples=500,
)
print(f"Strategy 1: {len(query_samples)} samples")

In [ ]:
# Preview samples
import random
for s in random.sample(query_samples, min(3, len(query_samples))):
    print(f"[{s.category}] Q: {s.question}")
    print(f"  Context: {s.context[:100]}...")
    print(f"  Answer: {s.response[:150]}...")
    print()

## 2. Strategy 2: From Document Chunks

In [ ]:
from finetuning.response_model.generate_training_data import generate_from_chunks

chunk_samples = generate_from_chunks(
    chunks_dir=CHUNKS_DIR,
    api_key=GEMINI_API_KEY,
    max_samples=200,
)
print(f"Strategy 2: {len(chunk_samples)} samples")

## 3. Validate with LLM

In [ ]:
from finetuning.response_model.generate_training_data import validate_with_llm

all_samples = query_samples + chunk_samples
print(f"Total before validation: {len(all_samples)}")

valid_samples, rejected = validate_with_llm(all_samples, GEMINI_API_KEY)
print(f"Valid: {len(valid_samples)}, Rejected: {len(rejected)}")

if rejected:
    print("\nFirst 3 rejections:")
    for r in rejected[:3]:
        print(f"  Q: {r['question'][:60]}")
        print(f"  Reason: {r['reason']}")
        print()

## 4. Save

In [ ]:
from finetuning.response_model.generate_training_data import (
    save_to_csv, save_prompt_template_csv, _contains_legal_reference
)

train_path, val_path = save_to_csv(valid_samples, OUTPUT_DIR)
prompt_path = save_prompt_template_csv(OUTPUT_DIR)

# Stats
categories = {}
ref_count = 0
for s in valid_samples:
    categories[s.category] = categories.get(s.category, 0) + 1
    if _contains_legal_reference(s.response):
        ref_count += 1

print(f"\nSummary: {len(valid_samples)} samples")
for cat, cnt in sorted(categories.items()):
    print(f"  {cat}: {cnt}")
print(f"With legal references: {ref_count}/{len(valid_samples)} ({ref_count/max(len(valid_samples),1):.0%})")

In [ ]:
driver.close()
print("Done!")